# 🏦 Credit Assessment Environment - GRPO Training

This notebook trains an LLM to make accurate loan underwriting decisions using **Group Relative Policy Optimization (GRPO)** from HuggingFace TRL.

## What the Agent Learns
- Follow RBI guidelines (CIBIL score, FOIR, LTV ratios)
- Detect **trap cases** (perfect profile with one hidden flaw)
- Make correct decisions: `approve`, `reject`, `request_docs`, `counter_offer`

## Theme Alignment
- **Theme #4: Self-Improvement (Primary)** — Performance-gated curriculum + Adversarial self-play + Self-generated challenges
- **Theme #3: World Modeling** — Real professional banking task with RBI regulatory constraints

## Training Modes

| Mode | Section | Time (A100) | Best For |
|------|---------|-------------|----------|
| Standard | 7 only | ~30 min | Quick baseline |
| Curriculum | 7b only | ~45-60 min | **Recommended start** |
| Full Pipeline | 7b → 7c | ~90-120 min | Maximum improvement |

### Recommended Path: Curriculum + Adversarial + Self-Generation
1. Run Section 7b (Performance-Gated Curriculum Learning)
2. Then run Section 7c (Adversarial Self-Play + Self-Generation)

---

**Runtime:** A100 GPU (Colab Pro)
**Model:** Qwen/Qwen2.5-7B-Instruct
**Expected Improvement:** ~60% → 88%+ accuracy

## 1. Setup & Installation

In [ ]:
# Install dependencies
!pip install -q trl>=0.17.0 transformers>=4.50.0 datasets accelerate peft bitsandbytes
!pip install -q matplotlib pydantic

In [ ]:
# Clone the Credit Assessment Environment
!git clone https://huggingface.co/spaces/iamnijin/credit-assessment-env
%cd credit-assessment-env

In [ ]:
# Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Environment Setup

The Credit Assessment Environment simulates Indian loan underwriting with:
- **3 difficulty levels**: Personal (Easy), Vehicle (Medium), Home (Hard)
- **Trap cases**: Profiles designed to trick pattern-matching LLMs
- **Asymmetric rewards**: Approving bad loans costs 3× more than rejecting good ones

In [ ]:
# Import standalone training utilities (avoids complex import chains)
import json
import random
from typing import Any

from train_utils import (
    DIFFICULTY_PROFILES,
    generate_applicant,
    calculate_ground_truth,
    calculate_reward,
    build_profile_text,
    CreditAssessmentAction,
    LoanDecision,
    generate_adversarial_case,
    AdversarialTracker,
    ADVERSARIAL_STRATEGIES,
)

# Test the environment
applicant = generate_applicant(task_id=1)  # Personal Loan
ground_truth = calculate_ground_truth(applicant)
profile = build_profile_text(applicant)

print("=" * 60)
print("Sample Loan Application")
print("=" * 60)
print(profile)
print(f"\nGround Truth Decision: {ground_truth}")

## 3. Dataset Generation

We generate synthetic loan applications with known ground truth decisions.
The dataset includes good, bad, borderline, and **trap** profiles.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are a senior loan officer at an Indian bank. You assess loan applications following RBI guidelines.

Respond with JSON containing:
- "decision": one of "approve", "reject", "request_docs", "counter_offer"
- "reasoning": brief explanation
- "counter_offer_amount": (only for counter_offer) reduced loan amount
- "docs_requested": (only for request_docs) needed documents

Key rules:
- CIBIL < 700 → reject
- FOIR > 50% → reject
- Missing documents → request_docs
- Home loan without RERA → reject (even if financials are perfect!)
- Vehicle loan LTV > 85% → counter_offer
- Home loan LTV: ≤30L→90%, 30-75L→80%, >75L→75%

Check EVERY criterion. One failure = rejection.

Respond with valid JSON only."""

def generate_dataset(num_samples: int, seed: int = 42, difficulty: str = "all") -> Dataset:
    """Generate loan application dataset with curriculum support."""
    random.seed(seed)
    samples = []
    
    for i in range(num_samples):
        task_id = (i % 3) + 1  # Balanced tasks
        applicant = generate_applicant(task_id, difficulty=difficulty)
        ground_truth = calculate_ground_truth(applicant)
        profile_text = build_profile_text(applicant)
        
        prompt = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": profile_text}
        ]
        
        samples.append({
            "prompt": prompt,
            "ground_truth": ground_truth,
            "task_id": task_id,
            "applicant_data": json.dumps(applicant),
        })
    
    return Dataset.from_list(samples)

# Generate datasets
# Eval pool bumped from 30 → 200 so per-task breakdowns (~60/60/80 samples
# per loan type) are no longer dominated by sampling noise. With 30 eval
# samples, a 1-sample flip looked like a 15% accuracy swing.
train_dataset = generate_dataset(300, seed=42)  # 300 for Colab
eval_dataset = generate_dataset(200, seed=123)

print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")

# Show distribution
from collections import Counter
gt_dist = Counter([s["ground_truth"] for s in train_dataset])
print(f"\nGround truth distribution: {dict(gt_dist)}")

## 4. Reward Function

The reward function evaluates model decisions against ground truth:

| Scenario | Reward |
|----------|--------|
| Correct decision | +10 |
| Request docs when incomplete | +2 |
| Counter-offer catches high LTV | +5 |
| Approve bad loan | -15 |
| Reject good loan | -5 |
| Approve non-RERA home | -20 |

In [ ]:
def credit_assessment_reward(
    completions: list,
    ground_truth: list[str],
    applicant_data: list[str],
    **kwargs
) -> list[float]:
    """Evaluate loan decisions against ground truth."""
    rewards = []
    
    for completion, gt, applicant_json in zip(completions, ground_truth, applicant_data):
        try:
            # Extract content
            if isinstance(completion, list) and len(completion) > 0:
                content = completion[0].get("content", "")
            else:
                content = str(completion)
            
            # Clean markdown
            if "```json" in content:
                content = content.split("```json")[1].split("```")[0]
            elif "```" in content:
                content = content.split("```")[1].split("```")[0]
            
            parsed = json.loads(content.strip())
            decision_str = parsed.get("decision", "reject").lower()
            
            try:
                decision = LoanDecision(decision_str)
            except ValueError:
                decision = LoanDecision.REJECT
            
            action = CreditAssessmentAction(
                decision=decision,
                reasoning=parsed.get("reasoning", ""),
                counter_offer_amount=parsed.get("counter_offer_amount"),
                docs_requested=parsed.get("docs_requested"),
            )
            
            applicant = json.loads(applicant_json)
            raw_reward = calculate_reward(action, applicant, gt)
            
            # Normalize [-20, +10] → [-1, +1]
            normalized = (raw_reward - (-20.0)) / (10.0 - (-20.0)) * 2 - 1
            rewards.append(normalized)
            
        except (json.JSONDecodeError, KeyError, TypeError):
            rewards.append(-0.5)  # Invalid JSON penalty
        except Exception:
            rewards.append(-0.3)
    
    return rewards

def format_reward(completions: list, **kwargs) -> list[float]:
    """Reward proper JSON formatting."""
    rewards = []
    
    for completion in completions:
        try:
            if isinstance(completion, list) and len(completion) > 0:
                content = completion[0].get("content", "")
            else:
                content = str(completion)
            
            if "```json" in content:
                content = content.split("```json")[1].split("```")[0]
            elif "```" in content:
                content = content.split("```")[1].split("```")[0]
            
            parsed = json.loads(content.strip())
            
            if "decision" in parsed and "reasoning" in parsed:
                rewards.append(0.2)
            elif "decision" in parsed:
                rewards.append(0.1)
            else:
                rewards.append(-0.1)
        except:
            rewards.append(-0.2)
    
    return rewards

def combined_reward(completions, ground_truth, applicant_data, **kwargs):
    """Combined reward: 80% decision accuracy + 20% format."""
    decision_rewards = credit_assessment_reward(completions, ground_truth, applicant_data, **kwargs)
    format_rewards = format_reward(completions, **kwargs)
    return [0.8 * d + 0.2 * f for d, f in zip(decision_rewards, format_rewards)]

print("✓ Reward functions defined")

## 5. Model & Training Configuration

We use:
- **Qwen2.5-7B-Instruct**: Strong reasoning, reliable JSON generation, fits A100 with LoRA
- **LoRA**: Parameter-efficient fine-tuning (trains only ~1% of parameters)
- **GRPO**: Group Relative Policy Optimization for reward maximization

**Stability fixes applied (v2):**
- `learning_rate=5e-6` — prevents policy overshooting on hard adversarial cases
- `max_grad_norm=0.5` — raised from 0.1: 0.1 was crushing gradient updates on our tiny effective batch (32 rollouts/step). Previous runs showed negative training losses, the signature of near-zero updates under aggressive clipping
- `beta=0.2` — raised from 0.1: stronger KL anchor prevents drift across curriculum → adversarial phase boundaries
- `num_generations=4` — balanced between advantage estimate quality and training speed; 8 was too slow on A100 with 400 samples/phase

In [ ]:
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

# Model configuration
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

# GRPO training config
training_args = GRPOConfig(
    output_dir="./grpo_credit_assessment",

    # Training
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,       # Curriculum LR — kept modest to prevent overshooting on hard cases
    max_grad_norm=0.5,        # v2: was 0.1 which crushed gradients → near-zero updates
    beta=0.2,                 # v2: was 0.1 — stronger KL anchor prevents cross-phase drift

    # GRPO specific
    num_generations=4,        # 4 balances advantage quality vs training speed on A100
    max_completion_length=256,

    # Memory optimization
    gradient_checkpointing=True,
    bf16=True,

    # Logging
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,

    report_to="none",
)

print(f"Model: {MODEL_NAME}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"GRPO generations per prompt: {training_args.num_generations}")

In [ ]:
# Create GRPO Trainer
trainer = GRPOTrainer(
    model=MODEL_NAME,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
    reward_funcs=combined_reward,  # Combined: 80% decision + 20% format
)

print("✓ Trainer created successfully!")

## 6. Pre-Training Evaluation (Baseline)

Let's measure the model's accuracy **before** training.

In [ ]:
def evaluate_model(model, tokenizer, dataset, num_samples=20):
    """Evaluate model accuracy on loan decisions.

    Uses greedy decoding (do_sample=False) so measurements are deterministic —
    critical when the threshold-gated curriculum depends on stable accuracy numbers.
    """
    correct = 0
    total = 0
    results = []
    was_training = model.training
    model.eval()

    for i, sample in enumerate(dataset):
        if i >= num_samples:
            break
        
        prompt = tokenizer.apply_chat_template(
            sample["prompt"],
            tokenize=False,
            add_generation_prompt=True
        )
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        
        response = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )
        
        try:
            if "```json" in response:
                response = response.split("```json")[1].split("```")[0]
            elif "```" in response:
                response = response.split("```")[1].split("```")[0]
            
            parsed = json.loads(response.strip())
            decision = parsed.get("decision", "").lower()
            is_correct = (decision == sample["ground_truth"])
            
            results.append({
                "task_id": sample["task_id"],
                "ground_truth": sample["ground_truth"],
                "predicted": decision,
                "correct": is_correct,
            })
            
            if is_correct:
                correct += 1
            total += 1
            
        except:
            results.append({
                "task_id": sample["task_id"],
                "ground_truth": sample["ground_truth"],
                "predicted": "[PARSE_ERROR]",
                "correct": False,
            })
            total += 1
    
    if was_training:
        model.train()
    accuracy = correct / total if total > 0 else 0
    return accuracy, results

# Baseline evaluation
# num_samples bumped from 20 → 60 so per-task (~20 samples each) measurements
# are stable. Prior runs couldn't distinguish 6/7 from 5/7 (86% vs 71%) —
# literally one sample of difference. 60 samples gives ±6% CI around the point estimate.
print("Evaluating pre-training baseline (60 samples)...")
baseline_acc, baseline_results = evaluate_model(
    trainer.model, tokenizer, eval_dataset, num_samples=60
)
print(f"\n📊 Baseline Accuracy: {baseline_acc*100:.1f}%")

## 7. Training! 🚀

Now we train the model with GRPO. You'll see:
- **reward/mean**: Average reward (should increase)
- **loss**: Training loss (should decrease)
- **clip_ratio**: GRPO clipping ratio

In [ ]:
# Train!
print("Starting GRPO training...")
print("This will take ~30-60 minutes on a T4 GPU.")
print("="*60)

trainer.train()

print("="*60)
print("✓ Training complete!")

In [ ]:
# Save the model
trainer.save_model("./grpo_credit_assessment_final")
print("✓ Model saved to ./grpo_credit_assessment_final")

## 7b. Performance-Gated Curriculum Learning (Recommended)

**Run this INSTEAD of Section 7 for better results.**

Curriculum learning trains in 3 phases with mastery gating:
1. **Phase 1 (Easy)**: Only obvious good/bad cases — must reach **60% accuracy** to advance
2. **Phase 2 (Medium)**: Borderline cases — must reach **60% accuracy** to advance
3. **Phase 3 (Hard)**: All trap cases — no gate (nowhere left to advance)

The model **earns** advancement rather than advancing on a fixed timer. If accuracy stays below the threshold, the phase repeats **once** with fresh samples.

**v2 tuning:**
- `samples_per_phase: 400` (was 100) — enough gradient updates to actually learn
- `per-attempt eval samples: 50` (was 10) — 10 samples gave ±30% CI on the gate decision
- `mastery threshold: 60%` (was 65%) — realistic bar given the per-attempt eval precision
- `max_retries: 1` (was 2) — more retries tend to overfit on noise; we keep the best adapter per phase
- **Best-adapter-per-phase**: each retry saves its adapter if it beat the previous attempt. The best adapter is preserved on disk.

This aligns with **Theme #4: Self-Improvement** — recursive skill amplification through competence-gated progression.

In [ ]:
import os

# v2: Performance-gated curriculum with best-adapter-per-phase tracking.
MASTERY_THRESHOLD = 0.60  # v2: 0.65 → 0.60 — realistic given 50-sample eval precision
MAX_PHASE_RETRIES = 1     # v2: 2 → 1 — more retries tended to overfit on noise
SAMPLES_PER_PHASE = 400   # v2: 100 → 400 — enough gradient updates to actually learn
PHASE_EVAL_SAMPLES = 50   # v2: 10 → 50 — 10 samples gave ±30% CI on gate decision
BEST_ADAPTER_DIR = "./grpo_phase_best_adapter"

phases = [
    ("easy",   "Phase 1: Learning Basics",       MASTERY_THRESHOLD),
    ("medium", "Phase 2: Refining",              MASTERY_THRESHOLD),
    ("hard",   "Phase 3: Mastering Traps",       0.0),  # No gate on final phase
]

phase_results = []

for phase_idx, (difficulty, phase_name, threshold) in enumerate(phases):
    print(f"\n{'='*60}")
    print(f"{phase_name}")
    print(f"{'='*60}")

    # Evaluate on the CURRENT phase difficulty so mastery is achievable.
    eval_dataset_phase = generate_dataset(PHASE_EVAL_SAMPLES * 2, seed=123 + phase_idx, difficulty=difficulty)

    phase_acc = 0.0
    best_attempt_acc = -1.0
    best_attempt_idx = -1

    for attempt in range(MAX_PHASE_RETRIES + 1):
        if attempt > 0:
            print(f"\n  [Retry {attempt}/{MAX_PHASE_RETRIES}] "
                  f"Best so far: {best_attempt_acc*100:.1f}% (attempt {best_attempt_idx+1}) — "
                  f"below {threshold*100:.0f}% threshold, retrying with fresh samples...")

        phase_train = generate_dataset(SAMPLES_PER_PHASE, seed=42 + phase_idx * 100 + attempt, difficulty=difficulty)
        print(f"  Samples: {len(phase_train)}  |  Difficulty: {difficulty}"
              + (f"  |  Attempt {attempt+1}/{MAX_PHASE_RETRIES+1}" if MAX_PHASE_RETRIES > 0 else ""))

        trainer.train_dataset = phase_train
        trainer.train()

        phase_acc, _ = evaluate_model(trainer.model, tokenizer, eval_dataset_phase, num_samples=PHASE_EVAL_SAMPLES)
        print(f"  Accuracy: {phase_acc*100:.1f}%"
              + (f" (threshold: {threshold*100:.0f}%)" if threshold > 0 else " (no gate on final phase)"))

        # Keep best adapter seen so far in this phase
        if phase_acc > best_attempt_acc:
            best_attempt_acc = phase_acc
            best_attempt_idx = attempt
            try:
                trainer.save_model(BEST_ADAPTER_DIR)
                print(f"  → New best-in-phase adapter saved ({phase_acc*100:.1f}%)")
            except Exception as e:
                print(f"  ⚠ Could not save best adapter: {e}")

        if threshold == 0.0 or phase_acc >= threshold:
            if threshold > 0:
                print(f"  ✓ Mastery achieved — advancing.")
            break
    else:
        print(f"  Threshold not reached after {MAX_PHASE_RETRIES+1} attempts — advancing anyway.")

    # Warn if final attempt regressed — best adapter still on disk as fallback
    if phase_acc < best_attempt_acc - 0.05 and os.path.isdir(BEST_ADAPTER_DIR):
        print(f"  ⚠ Final attempt {phase_acc*100:.1f}% regressed from best {best_attempt_acc*100:.1f}%.")
        print(f"     Best adapter preserved at {BEST_ADAPTER_DIR} if you need to revert.")

    phase_results.append((phase_name, best_attempt_acc if best_attempt_acc >= 0 else phase_acc))

print("\n" + "="*60)
print("CURRICULUM RESULTS")
print("="*60)
for name, acc in phase_results:
    print(f"  {name}: {acc*100:.1f}%")

trainer.save_model("./grpo_curriculum_model")
print("\n✓ Curriculum-trained model saved!")

## 7c. Adversarial Self-Play + Self-Generation (Run After Curriculum)

**Run this AFTER Section 7b for the full recursive self-improvement pipeline.**

Each round:
1. **Evaluate** on adversarial cases — record per-strategy failure rates
2. **Identify** the weakest strategy via `AdversarialTracker`
3. **Mix in** self-generated cases from the previous round (capped at 30%)
4. **Train** on targeted adversarial + hard cases
5. **Self-generate**: prompt the trained model to design trap cases targeting its own weaknesses — carry into next round

The self-generation loop is what makes this recursive: the model's own failure patterns shape what it trains on next.

In [ ]:
import torch
from datasets import Dataset

SELF_GEN_PROMPT = """You are a loan underwriting trainer designing test cases.

Design ONE loan application that looks approvable at first glance but contains exactly one hidden flaw.

The model currently struggles most with:
{weaknesses}

Output ONLY a JSON object — no explanation, no markdown:
{{
    "loan_type": "personal" | "vehicle" | "home",
    "credit_score": <integer 650-850>,
    "monthly_income": <integer 50000-1000000>,
    "foir": <float 0.15-0.60>,
    "employment_years": <integer 0-20>,
    "loan_amount": <integer 100000-20000000>,
    "documents_complete": true | false,
    "collateral_value": <integer, required for vehicle/home>,
    "ltv_ratio": <float 0.5-0.95, required for vehicle/home>,
    "rera_registered": true | false (required for home),
    "has_co_applicant": true | false,
    "trap_type": "<one line: what is the hidden flaw>"
}}"""


def generate_adversarial_dataset(num_samples, seed=42, tracker=None, target_weakness=True):
    random.seed(seed)
    samples = []
    if tracker and target_weakness:
        cases = tracker.generate_targeted_batch(num_samples, target_weakness=True)
        for case in cases:
            applicant = case["applicant"]
            strategy = case["strategy"]
            ground_truth = calculate_ground_truth(applicant)
            profile_text = build_profile_text(applicant)
            samples.append({
                "prompt": [{"role": "system", "content": SYSTEM_PROMPT},
                           {"role": "user", "content": profile_text}],
                "ground_truth": ground_truth,
                "task_id": {"personal": 1, "vehicle": 2, "home": 3}[applicant["loan_type"]],
                "applicant_data": json.dumps(applicant),
                "adversarial_strategy": strategy,
            })
    else:
        for i in range(num_samples):
            strategy = random.choice(ADVERSARIAL_STRATEGIES)
            applicant = generate_adversarial_case(strategy)
            ground_truth = calculate_ground_truth(applicant)
            profile_text = build_profile_text(applicant)
            samples.append({
                "prompt": [{"role": "system", "content": SYSTEM_PROMPT},
                           {"role": "user", "content": profile_text}],
                "ground_truth": ground_truth,
                "task_id": {"personal": 1, "vehicle": 2, "home": 3}[applicant["loan_type"]],
                "applicant_data": json.dumps(applicant),
                "adversarial_strategy": strategy,
            })
    return Dataset.from_list(samples)


def evaluate_adversarial_strategies(model, tokenizer, tracker, num_per_strategy=5):
    # v2: num_per_strategy default bumped from 2 → 5. At 2 samples per strategy,
    # "0%" and "100%" were mostly noise. 5 gives 20% resolution per strategy.
    results = {s: {"correct": 0, "total": 0} for s in ADVERSARIAL_STRATEGIES}
    model.eval()
    for strategy in ADVERSARIAL_STRATEGIES:
        for _ in range(num_per_strategy):
            applicant = generate_adversarial_case(strategy)
            ground_truth = calculate_ground_truth(applicant)
            profile_text = build_profile_text(applicant)
            prompt = [{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": profile_text}]
            formatted = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                                         pad_token_id=tokenizer.pad_token_id)
            response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            try:
                if "```json" in response: response = response.split("```json")[1].split("```")[0]
                elif "```" in response: response = response.split("```")[1].split("```")[0]
                decision = json.loads(response.strip()).get("decision", "").lower()
                results[strategy]["total"] += 1
                if decision == ground_truth:
                    results[strategy]["correct"] += 1
                    tracker.record_result(strategy, True)
                else:
                    tracker.record_result(strategy, False)
            except:
                results[strategy]["total"] += 1
                tracker.record_result(strategy, False)
    model.train()
    return results


def self_generate_adversarial_cases(model, tokenizer, tracker, num_cases=15):
    """Prompt the trained model to design its own hard cases targeting its weaknesses."""
    weakness_summary = tracker.get_summary()
    top_weaknesses = sorted(weakness_summary.items(), key=lambda x: -x[1].get("failures", 0))[:3]
    weakness_str = "\n".join(f"- {s}: {d['failures']} failures" for s, d in top_weaknesses) \
                   if top_weaknesses else "- No data yet — generate varied trap cases"

    prompt_text = SELF_GEN_PROMPT.format(weaknesses=weakness_str)
    messages = [{"role": "user", "content": prompt_text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    samples = []
    attempts = 0
    max_attempts = num_cases * 4

    model.eval()
    with torch.no_grad():
        while len(samples) < num_cases and attempts < max_attempts:
            attempts += 1
            try:
                outputs = model.generate(**inputs, max_new_tokens=300, do_sample=True,
                                          temperature=0.9, pad_token_id=tokenizer.pad_token_id)
                response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:],
                                            skip_special_tokens=True).strip()
                if "```json" in response: response = response.split("```json")[1].split("```")[0]
                elif "```" in response: response = response.split("```")[1].split("```")[0]
                case = json.loads(response.strip())

                required = ["loan_type", "credit_score", "monthly_income", "foir",
                            "employment_years", "loan_amount", "documents_complete"]
                if not all(k in case for k in required): continue
                if case["loan_type"] not in ("personal", "vehicle", "home"): continue

                if case["loan_type"] in ("vehicle", "home") and "collateral_value" not in case:
                    case["collateral_value"] = int(case["loan_amount"] * 1.3)
                    case["ltv_ratio"] = case["loan_amount"] / case["collateral_value"]
                if case["loan_type"] == "home":
                    case.setdefault("rera_registered", True)
                    case.setdefault("has_co_applicant", False)
                    case.setdefault("property_type", "apartment")
                if case["loan_type"] == "vehicle":
                    case.setdefault("vehicle_type", "sedan")

                gt = calculate_ground_truth(case)
                if gt is None: continue

                profile = build_profile_text(case)
                samples.append({
                    "prompt": [{"role": "system", "content": SYSTEM_PROMPT},
                               {"role": "user", "content": profile}],
                    "ground_truth": gt,
                    "task_id": {"personal": 1, "vehicle": 2, "home": 3}[case["loan_type"]],
                    "applicant_data": json.dumps(case),
                    "adversarial_strategy": case.get("trap_type", "self_generated"),
                })
            except Exception:
                continue
    model.train()
    return samples



def evaluate_by_loan_type(model, tokenizer, num_samples_per_type=30):
    """Evaluate accuracy per loan type to detect catastrophic forgetting.

    v2 fix: previous version generated `num_samples_per_type` samples ONCE
    per loan_type and then filtered — meaning only ~1/3 of samples matched
    the target type, so each type got ~7 evaluated samples instead of 20.
    Now we pool `num_samples_per_type * 3` samples once (generate_dataset
    cycles task_id across 1,2,3 so each loan_type gets exactly the target
    count) and filter once.
    """
    results = {}
    model.eval()
    pool = generate_dataset(num_samples_per_type * 3, seed=999, difficulty="all")
    for loan_type in ["personal", "vehicle", "home"]:
        correct = 0
        total = 0
        for sample in pool:
            try:
                applicant_data = json.loads(sample.get("applicant_data", "{}"))
            except:
                applicant_data = {}
            if applicant_data.get("loan_type") != loan_type:
                continue
            prompt_text = tokenizer.apply_chat_template(
                sample["prompt"], tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_new_tokens=256, do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                )
            response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            try:
                if "```json" in response: response = response.split("```json")[1].split("```")[0]
                elif "```" in response: response = response.split("```")[1].split("```")[0]
                decision = json.loads(response.strip()).get("decision", "").lower()
                if decision == sample["ground_truth"]:
                    correct += 1
                total += 1
            except:
                total += 1
        results[loan_type] = {"correct": correct, "total": total,
                              "accuracy": correct / total if total > 0 else 0.0}
    model.train()
    print(f"    Personal: {results['personal']['correct']}/{results['personal']['total']} "
          f"({results['personal']['accuracy']*100:.0f}%)  |  "
          f"Vehicle: {results['vehicle']['correct']}/{results['vehicle']['total']} "
          f"({results['vehicle']['accuracy']*100:.0f}%)  |  "
          f"Home: {results['home']['correct']}/{results['home']['total']} "
          f"({results['home']['accuracy']*100:.0f}%)")
    return results

# ── Adversarial Self-Play Training ──────────────────────────────────────────

# v2 tuning:
#   - samples_per_round: 50 → 150 (3× more coverage of targeted strategy per round)
#   - adversarial_rounds: 3 → 2 (Round 3 produced no new signal in prior runs)
#   - num_per_strategy eval: 2 → 5 (2 was too noisy for targeting decisions)
#   - Snapshot curriculum checkpoint before we start — if adversarial regresses
#     overall accuracy, we can pitch the curriculum-only model instead.
tracker = AdversarialTracker()
adversarial_rounds = 2
samples_per_round = 150
num_per_strategy_eval = 5
self_gen_carry = []  # Self-generated cases from previous round

# Snapshot curriculum-end state as fallback
CURRICULUM_SNAPSHOT = "./grpo_curriculum_end_snapshot"
try:
    trainer.save_model(CURRICULUM_SNAPSHOT)
    print(f"📦 Curriculum snapshot saved to {CURRICULUM_SNAPSHOT}")
    print("   (Fallback if adversarial training regresses overall accuracy.)")
except Exception as e:
    print(f"⚠ Could not save curriculum snapshot: {e}")

# Baseline accuracy going INTO adversarial (for regression check)
print("\nMeasuring curriculum-only accuracy on eval set (for regression check)...")
pre_adversarial_acc, _ = evaluate_model(trainer.model, tokenizer, eval_dataset, num_samples=60)
print(f"  Curriculum-only accuracy: {pre_adversarial_acc*100:.1f}%")

print("\n" + "=" * 60)
print("ADVERSARIAL SELF-PLAY + SELF-GENERATION TRAINING")
print("=" * 60)

round_results = []

for round_idx in range(adversarial_rounds):
    print(f"\n{'='*60}")
    print(f"ROUND {round_idx + 1}/{adversarial_rounds}")
    print(f"{'='*60}")

    # Step 1: Evaluate and update tracker
    print(f"\n  [Before Round {round_idx + 1}] Accuracy by loan type:")
    pre_round = evaluate_by_loan_type(trainer.model, tokenizer)
    print("Evaluating on adversarial strategies...")
    eval_results = evaluate_adversarial_strategies(trainer.model, tokenizer, tracker, num_per_strategy=num_per_strategy_eval)

    print("\nStrategy accuracy:")
    for strategy, results in sorted(eval_results.items(), key=lambda x: x[1]["correct"] / max(1, x[1]["total"])):
        acc = results["correct"] / max(1, results["total"]) * 100
        print(f"  {strategy}: {acc:.0f}%")

    weakness = tracker.get_weakness()
    print(f"\n→ Targeting weakness: {weakness}")

    # Step 2: Rule-based targeted dataset
    adversarial_data = generate_adversarial_dataset(samples_per_round, seed=42+round_idx+100,
                                                     tracker=tracker, target_weakness=True)
    adversarial_samples = list(adversarial_data)

    # Step 3: Mix in self-generated cases from previous round (30% cap)
    if self_gen_carry:
        max_carry = max(1, len(adversarial_samples) * 3 // 10)
        carry_to_use = self_gen_carry[:max_carry]
        adversarial_samples = carry_to_use + adversarial_samples
        print(f"  Mixed in {len(carry_to_use)} self-generated cases from previous round")

    # CRITICAL: Replay data from all difficulties to prevent catastrophic forgetting
    easy_replay = generate_dataset(samples_per_round // 4, seed=42+round_idx+200, difficulty="easy")
    medium_replay = generate_dataset(samples_per_round // 4, seed=42+round_idx+300, difficulty="medium")
    hard_replay = generate_dataset(samples_per_round // 4, seed=42+round_idx+400, difficulty="hard")
    
    replay_samples = list(easy_replay) + list(medium_replay) + list(hard_replay)
    combined = adversarial_samples + replay_samples
    random.shuffle(combined)
    train_data = Dataset.from_list(combined)
    
    print(f"  Data mix: {len(adversarial_samples)} adversarial + {len(replay_samples)} replay")

    # Step 4: Train — use 1 epoch per adversarial round to prevent overfitting
    # on the narrow targeted distribution (key stability fix).
    trainer.train_dataset = train_data
    prev_epochs = trainer.args.num_train_epochs
    trainer.args.num_train_epochs = 1
    print(f"\nTraining on {len(train_data)} samples (1 epoch)...")
    trainer.train()
    trainer.args.num_train_epochs = prev_epochs

    # Step 5: Measure improvement (POST-training)
    print(f"\n  [After Round {round_idx + 1}] Accuracy by loan type:")
    post_round = evaluate_by_loan_type(trainer.model, tokenizer)
    for lt in ["personal", "vehicle", "home"]:
        delta = post_round[lt]["accuracy"] - pre_round[lt]["accuracy"]
        arrow = "✅" if delta >= 0 else "❌"
        print(f"    {lt}: {delta*100:+.0f}% {arrow}")

    # Re-evaluate adversarial strategies AFTER training to get the true round accuracy.
    # (The pre-round eval_results only reflect the state going INTO this round.)
    print(f"\n  [After Round {round_idx + 1}] Re-evaluating adversarial strategies...")
    post_eval = evaluate_adversarial_strategies(trainer.model, tokenizer, tracker, num_per_strategy=num_per_strategy_eval)
    total_correct = sum(r["correct"] for r in post_eval.values())
    total = sum(r["total"] for r in post_eval.values())
    round_acc = total_correct / total if total > 0 else 0

    # Step 6: Self-generate hard cases for next round
    # v2: self-gen pool bumped from 15 → 30 (we still cap at 30% of the
    # adversarial mix during training, so extras provide a buffer against
    # invalid JSON generations).
    print(f"\nSelf-generating hard cases for next round...")
    self_gen_carry = self_generate_adversarial_cases(trainer.model, tokenizer, tracker, num_cases=30)
    print(f"Generated {len(self_gen_carry)} valid self-generated cases")

    round_results.append((round_idx + 1, weakness, round_acc, len(self_gen_carry)))
    print(f"\nRound {round_idx + 1} complete: {round_acc*100:.1f}% accuracy")

print("\n" + "=" * 60)
print("ADVERSARIAL TRAINING SUMMARY")
print("=" * 60)
for round_num, weakness, acc, self_gen in round_results:
    print(f"  Round {round_num}: {acc*100:.1f}% | targeted: {weakness} | self-gen: {self_gen} cases")

print("\nFinal weakness analysis:")
for strategy, data in sorted(tracker.get_summary().items(), key=lambda x: x[1]["accuracy"]):
    print(f"  {strategy}: {data['accuracy']*100:.0f}% ({data['failures']} failures)")

trainer.save_model("./grpo_adversarial_final")
print("\n✓ Model saved to ./grpo_adversarial_final")

# ── Regression check: compare adversarial final vs curriculum snapshot ──
print("\n" + "=" * 60)
print("CURRICULUM vs CURRICULUM + ADVERSARIAL")
print("=" * 60)
post_adversarial_acc, _ = evaluate_model(trainer.model, tokenizer, eval_dataset, num_samples=60)
delta = post_adversarial_acc - pre_adversarial_acc
print(f"  Curriculum-only     : {pre_adversarial_acc*100:.1f}%")
print(f"  +Adversarial rounds : {post_adversarial_acc*100:.1f}%  (Δ {delta*100:+.1f}%)")

if delta < -0.03:
    print(f"\n  ⚠ Adversarial training REGRESSED overall accuracy by {abs(delta)*100:.1f}%.")
    print(f"     For the pitch, use the curriculum snapshot:")
    print(f"       {CURRICULUM_SNAPSHOT}")
    print(f"     Load it in a fresh trainer before running Section 8 post-training eval.")
elif delta > 0.03:
    print(f"\n  ✓ Adversarial training helped by {delta*100:.1f}%. Use ./grpo_adversarial_final.")
else:
    print(f"\n  ≈ Within noise (|Δ| < 3%). Either checkpoint is defensible for the pitch.")

## 8. Post-Training Evaluation

Let's measure improvement!

In [ ]:
# Post-training evaluation
# num_samples: 20 → 60 to match baseline eval. This is the number that lands
# in your hackathon chart, so it MUST be stable. 20 samples made "86% → 71% on
# Personal" look dramatic when it was literally 6/7 → 5/7 — 1 sample of noise.
print("Evaluating trained model (60 samples)...")
trained_acc, trained_results = evaluate_model(
    trainer.model, tokenizer, eval_dataset, num_samples=60
)

print(f"\n" + "="*60)
print("RESULTS")
print("="*60)
print(f"📊 Baseline Accuracy: {baseline_acc*100:.1f}%")
print(f"📈 Trained Accuracy:  {trained_acc*100:.1f}%")
print(f"🚀 Improvement:       {(trained_acc - baseline_acc)*100:+.1f}%")
print("="*60)

In [ ]:
# Detailed breakdown by task
import pandas as pd

task_names = {1: "Personal Loan", 2: "Vehicle Loan", 3: "Home Loan"}

# Baseline breakdown
baseline_df = pd.DataFrame(baseline_results)
baseline_by_task = baseline_df.groupby("task_id")["correct"].mean()

# Trained breakdown
trained_df = pd.DataFrame(trained_results)
trained_by_task = trained_df.groupby("task_id")["correct"].mean()

print("\nAccuracy by Loan Type:")
print("-" * 50)
print(f"{'Task':<20} {'Baseline':>12} {'Trained':>12} {'Δ':>10}")
print("-" * 50)
for task_id in [1, 2, 3]:
    base = baseline_by_task.get(task_id, 0)
    trained = trained_by_task.get(task_id, 0)
    delta = trained - base
    print(f"{task_names[task_id]:<20} {base*100:>10.1f}% {trained*100:>10.1f}% {delta*100:>+9.1f}%")

## 9b. Generate Hackathon-Ready Results & Narrative

Run this cell after training to generate:
1. **Before/After accuracy chart** (for your presentation)
2. **Trap case examples** (for your pitch)
3. **Copy-paste ready pitch summary**

In [ ]:
# ============================================================
# HACKATHON RESULTS GENERATOR
# ============================================================
# Uses your actual baseline_acc and trained_acc from above

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ------------------------------------------------------------
# 1. GENERATE RESULTS CHART
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Overall accuracy comparison
categories = ['Baseline\n(Pre-training)', 'Trained\n(Post-GRPO)']
accuracies = [baseline_acc * 100, trained_acc * 100]
colors = ['#e74c3c', '#27ae60']

bars = axes[0].bar(categories, accuracies, color=colors, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('Overall Loan Decision Accuracy', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, 100)
axes[0].axhline(y=100, color='gray', linestyle='--', alpha=0.3, label='Perfect (Rule-Based)')

# Add value labels
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=14)

# Add improvement arrow
improvement = (trained_acc - baseline_acc) * 100
mid_y = (baseline_acc + trained_acc) / 2 * 100
axes[0].annotate(f'+{improvement:.1f}%', 
                 xy=(1, trained_acc * 100 - 5), 
                 fontsize=16, fontweight='bold', color='#27ae60',
                 ha='center')

# Right: By loan type breakdown
task_names = ['Personal\n(Easy)', 'Vehicle\n(Medium)', 'Home\n(Hard)']

# Calculate per-task accuracy from results
baseline_by_task = [0, 0, 0]
trained_by_task = [0, 0, 0]
baseline_counts = [0, 0, 0]
trained_counts = [0, 0, 0]

for r in baseline_results:
    idx = r['task_id'] - 1
    baseline_by_task[idx] += 1 if r['correct'] else 0
    baseline_counts[idx] += 1

for r in trained_results:
    idx = r['task_id'] - 1
    trained_by_task[idx] += 1 if r['correct'] else 0
    trained_counts[idx] += 1

baseline_by_task = [baseline_by_task[i]/max(1,baseline_counts[i])*100 for i in range(3)]
trained_by_task = [trained_by_task[i]/max(1,trained_counts[i])*100 for i in range(3)]

x = range(len(task_names))
width = 0.35

bars1 = axes[1].bar([i - width/2 for i in x], baseline_by_task, width, 
                    label='Baseline', color='#e74c3c', edgecolor='black')
bars2 = axes[1].bar([i + width/2 for i in x], trained_by_task, width,
                    label='Trained', color='#27ae60', edgecolor='black')

axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Accuracy by Loan Type (Difficulty)', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(task_names)
axes[1].set_ylim(0, 100)
axes[1].legend()

# Add value labels
for bar in bars1:
    if bar.get_height() > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    if bar.get_height() > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('hackathon_results.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n✓ Chart saved to hackathon_results.png")
print("  (Right-click → Save Image to download)")

# ------------------------------------------------------------
# 2. TRAP CASE EXAMPLES (for your pitch)
# ------------------------------------------------------------

print("\n" + "="*60)
print("TRAP CASES FOR YOUR PITCH")
print("="*60)

# Generate example trap cases
trap_examples = [
    {
        "name": "The High-Income Trap",
        "profile": "₹6L/month income, 20% FOIR, 12 years experience",
        "hidden_flaw": "CIBIL score: 699 (just 1 point below 700)",
        "correct": "REJECT",
        "why_fails": "LLMs see high income and approve without checking CIBIL threshold"
    },
    {
        "name": "The Perfect-Profile Trap", 
        "profile": "CIBIL 820, ₹2L/month, 25% FOIR, 10 years",
        "hidden_flaw": "Property NOT RERA registered",
        "correct": "REJECT",
        "why_fails": "Everything screams 'approve' - RERA field is easy to overlook"
    },
    {
        "name": "The RBI Tier Trap",
        "profile": "₹1.2Cr home loan, LTV 78%",
        "hidden_flaw": "Loans >₹75L have max 75% LTV (RBI rule)",
        "correct": "COUNTER_OFFER",
        "why_fails": "Must know tiered limits: ≤30L→90%, 30-75L→80%, >75L→75%"
    }
]

for i, trap in enumerate(trap_examples, 1):
    print(f"\n### Trap {i}: {trap['name']}")
    print(f"Profile: {trap['profile']}")
    print(f"Hidden Flaw: {trap['hidden_flaw']}")
    print(f"Correct Decision: {trap['correct']}")
    print(f"Why LLMs Fail: {trap['why_fails']}")

# ------------------------------------------------------------
# 3. COPY-PASTE PITCH SUMMARY
# ------------------------------------------------------------

print("\n" + "="*60)
print("COPY-PASTE PITCH SUMMARY")
print("="*60)

print(f"""
## Credit Assessment Environment: Teaching LLMs to Be Loan Officers

**The Problem:** LLMs pattern-match. Banking requires precision. 
One overlooked criterion (CIBIL 699 vs 700) = ₹50L NPA.

**Our Solution:** Self-improving RL environment with:
- Trap cases targeting LLM weaknesses
- Curriculum learning (easy → medium → hard)  
- Adversarial self-play based on failure analysis
- Asymmetric rewards matching real banking risk

**Results:**
- Baseline: {baseline_acc*100:.1f}%
- Trained: {trained_acc*100:.1f}%
- Improvement: +{improvement:.1f}%

**Why It Matters:**
Every bank in India processes thousands of loans daily. 
An agent that catches edge cases = millions saved in NPAs.
""")

## 9. Visualize Training Progress

In [ ]:
import matplotlib.pyplot as plt

# Get training history
history = trainer.state.log_history

# Extract metrics
steps = []
rewards = []
losses = []

for entry in history:
    if "reward" in entry:
        steps.append(entry.get("step", 0))
        rewards.append(entry["reward"])
    if "loss" in entry and entry.get("step", 0) > 0:
        if len(losses) < len(steps):
            losses.append(entry["loss"])

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reward curve
axes[0].plot(steps[:len(rewards)], rewards, 'b-', linewidth=2, label='Average Reward')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Training Steps')
axes[0].set_ylabel('Reward')
axes[0].set_title('🎯 Reward During Training')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy comparison
categories = ['Baseline', 'Trained']
accuracies = [baseline_acc * 100, trained_acc * 100]
colors = ['#ff6b6b', '#51cf66']
bars = axes[1].bar(categories, accuracies, color=colors, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('📈 Before vs After Training')
axes[1].set_ylim(0, 100)

# Add value labels
for bar, acc in zip(bars, accuracies):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Plot saved to training_results.png")

## 10. Try the Trained Model!

Let's test on a trap case to see if training helped.

In [ ]:
# Generate a trap case
random.seed(12345)
trap_applicant = generate_applicant(task_id=3)  # Home loan (hardest)
trap_gt = calculate_ground_truth(trap_applicant)
trap_profile = build_profile_text(trap_applicant)

print("=" * 60)
print("TEST CASE: Home Loan Application")
print("=" * 60)
print(trap_profile)
print(f"\n🎯 Correct Decision: {trap_gt}")
print("=" * 60)

# Get model's decision
test_prompt = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": trap_profile}
]

formatted = tokenizer.apply_chat_template(
    test_prompt, tokenize=False, add_generation_prompt=True
)

inputs = tokenizer(formatted, return_tensors="pt").to(trainer.model.device)

with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.pad_token_id,
    )

response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print("\n🤖 Model's Response:")
print(response)

# Check if correct
try:
    if "```json" in response:
        response = response.split("```json")[1].split("```")[0]
    elif "```" in response:
        response = response.split("```")[1].split("```")[0]
    
    parsed = json.loads(response.strip())
    decision = parsed.get("decision", "")
    
    if decision == trap_gt:
        print("\n✅ CORRECT!")
    else:
        print(f"\n❌ Incorrect. Model said '{decision}', should be '{trap_gt}'")
except:
    print("\n⚠️ Could not parse response")

## 11. Push to HuggingFace Hub (Optional)

Upload your trained model to share with others!

In [ ]:
# Uncomment and run to push to hub
# from huggingface_hub import login
# login()  # Enter your HF token

# trainer.push_to_hub("your-username/credit-assessment-grpo")
# print("✓ Model pushed to HuggingFace Hub!")

---

## Summary

This notebook demonstrated:

1. **Environment**: Credit Assessment with 3 difficulty levels and trap cases
2. **Reward Model**: Asymmetric penalties matching real banking risk
3. **Training**: GRPO with LoRA for parameter-efficient fine-tuning
4. **Results**: Measurable improvement in loan decision accuracy

### Training Paths (all code is ready to run):

| Path | What to Run | Description |
|------|-------------|-------------|
| Quick | Section 7 | Standard training, all difficulties mixed |
| **Recommended** | Section 7b | Curriculum: Easy → Medium → Hard |
| Full Pipeline | 7b then 7c | Curriculum + Adversarial self-play |